In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
from transformers import AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model = AutoModelForCausalLM.from_pretrained(model_name)

# Total parameters
total_params = sum(p.numel() for p in model.parameters())

# Trainable parameters (with LoRA, only injected adapters are trainable)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Percentage: {100 * trainable_params / total_params:.2f}%")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Total Parameters: 1,100,048,384
Trainable Parameters: 1,100,048,384
Percentage: 100.00%


You're seeing 100% trainable parameters because the model was loaded without any parameter-efficient fine-tuning (PEFT) method like LoRA. Let's demonstrate how to apply LoRA to make only a small subset of the model's parameters trainable.

In [8]:
# To fix the torchao version incompatibility, we need to upgrade torchao.
# Let's first uninstall the current version and then install a compatible one.
!pip uninstall torchao -y
!pip install torchao==0.17.0 -qqq

from peft import LoraConfig, get_peft_model

# Configuration for LoRA
lora_config = LoraConfig(
    r=8,  # LoRA attention dimension
    lora_alpha=16,  # Alpha parameter for LoRA scaling
    target_modules=["q_proj", "v_proj"], # Target specific layers for LoRA
    lora_dropout=0.05, # Dropout probability for LoRA layers
    bias="none", # Bias type
    task_type="CAUSAL_LM" # Task type
)

# Apply LoRA to the model
lora_model = get_peft_model(model, lora_config)

# Recalculate trainable parameters for the LoRA model
total_params_lora = sum(p.numel() for p in lora_model.parameters())
trainable_params_lora = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)

print(f"Total Parameters (with LoRA): {total_params_lora:,}")
print(f"Trainable Parameters (with LoRA): {trainable_params_lora:,}")
print(f"Percentage (with LoRA): {100 * trainable_params_lora / total_params_lora:.2f}%")

Found existing installation: torchao 0.17.0
Uninstalling torchao-0.17.0:
  Successfully uninstalled torchao-0.17.0


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Total Parameters (with LoRA): 1,101,174,784
Trainable Parameters (with LoRA): 1,126,400
Percentage (with LoRA): 0.10%


In [9]:
from transformers import AutoTokenizer

# Load tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Your dataset
data = {
    "question": [
        "What is diabetes?",
        "How to treat a common cold?",
        "What are the symptoms of hypertension?"
    ],
    "answer": [
        "Diabetes is a chronic disease where the body cannot regulate blood sugar properly.",
        "Rest, hydration, and over-the-counter medications can help manage symptoms.",
        "Symptoms include headaches, shortness of breath, and nosebleeds."
    ]
}

# Tokenize questions
encodings = tokenizer(
    data["question"],
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print("Input IDs:\n", encodings["input_ids"])
print("Attention Masks:\n", encodings["attention_mask"])


Input IDs:
 tensor([[    1,  1724,   338,   652,   370, 10778, 29973,     2,     2,     2,
             2],
        [    1,  1128,   304,  7539,   263,  3619, 11220, 29973,     2,     2,
             2],
        [    1,  1724,   526,   278, 25828,  4835,   310,  7498, 10700,  2673,
         29973]])
Attention Masks:
 tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


### Task 8: Fine-Tune Using LoRA

#### Why Base Model Weights Remain Frozen with LoRA

When using LoRA (Low-Rank Adaptation) for fine-tuning, the base model's weights remain frozen because LoRA introduces small, trainable matrices (adapters) alongside the original weights. These adapters are trained instead of the entire pre-trained model. This approach offers several key advantages:

1.  **Efficiency**: By training only a small fraction of parameters, LoRA significantly reduces computational costs and memory requirements compared to full fine-tuning.
2.  **Catastrophic Forgetting Prevention**: Freezing the base model weights helps prevent catastrophic forgetting, where the model might lose its general knowledge learned during pre-training when fine-tuned on a specific task.
3.  **Storage**: The trained adapters are much smaller than the full model, making them easier to store, share, and deploy.

During inference, these trained adapter weights are combined with the frozen pre-trained weights to make predictions, effectively adapting the model's behavior to the new task without altering its core knowledge.

### Task 9: Save LoRA Adapter

Now, let's save only the LoRA adapter weights. This is a key advantage of PEFT methods like LoRA, as the saved file will be much smaller than saving the entire fine-tuned model.

In [10]:
# Save only the LoRA adapter weights
lora_model.save_pretrained("tinyllama_lora_adapter")

print("LoRA adapter saved successfully to 'tinyllama_lora_adapter/'")

# You can verify the size of the saved adapter compared to the full model
# !du -sh tinyllama_lora_adapter/
# (Note: full model is >1GB, adapter is typically a few MB)

LoRA adapter saved successfully to 'tinyllama_lora_adapter/'


#### Why LoRA Adapters are Small

LoRA adapters are small because they implement a low-rank decomposition of the weight update matrices. Instead of directly learning full-rank updates for each weight matrix in the neural network, LoRA approximates these updates using two much smaller matrices. For a weight matrix $W_0 \in \mathbb{R}^{d \times k}$, LoRA approximates the update $\Delta W$ as $BA$, where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$, and $r$ is a much smaller rank ($r \ll \min(d, k)$).

This means that instead of learning $d \times k$ parameters for $\Delta W$, LoRA only needs to learn $(d \times r) + (r \times k)$ parameters. Since $r$ is typically a very small number (e.g., 8, 16, 32), the number of parameters to learn and store for the adapters is dramatically reduced compared to the original model's parameters or a full fine-tuning approach.

For example, if $d=1024$, $k=1024$, and $r=8$:
- Full update: $1024 \times 1024 = 1,048,576$ parameters.
- LoRA update: $(1024 \times 8) + (8 \times 1024) = 8192 + 8192 = 16,384$ parameters.

This significant reduction in parameters is why LoRA adapters are lightweight and efficient.

### Task 10: Test LoRA Adapter

Now, let's load the saved LoRA adapter and use the model to generate some text.

In [11]:
from peft import PeftModel

# Load the base model again (if not already in scope from earlier cells)
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# base_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# Assuming `model` from earlier cell is the base model

# Load the LoRA adapter onto the base model
peft_model_loaded = PeftModel.from_pretrained(model, "tinyllama_lora_adapter")

print("LoRA adapter loaded successfully!")

LoRA adapter loaded successfully!


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Let's generate some text using the model with the loaded LoRA adapter.

In [12]:
prompt = "What is diabetes?"

# Tokenize the input prompt
inputs = tokenizer(prompt, return_tensors="pt").to(peft_model_loaded.device)

# Generate a response
output_tokens = peft_model_loaded.generate(**inputs, max_new_tokens=50, do_sample=True, top_k=50, top_p=0.95)

# Decode and print the response
response = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
print(response)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is diabetes? And why do you need to protect yourself from the dangers of sun exposure?
